In [7]:
import numpy as np
import pandas as pd
import os
from data_preprocessor import*
from py_vollib.black_scholes import black_scholes as bs
from py_vollib.black_scholes.greeks.analytical import delta, gamma, vega, theta, rho

In [4]:
df = merge_data('Options_AAPL_Monthly.csv','AAPL.CSV')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 683323 entries, 0 to 683322
Data columns (total 17 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Ticker                683323 non-null  object 
 1   TradeDate             683323 non-null  int64  
 2   Strike                683323 non-null  float64
 3   ExpiryDate            683323 non-null  int64  
 4   CallPut               683323 non-null  object 
 5   LastTradePrice        683323 non-null  float64
 6   BidPrice              683323 non-null  float64
 7   AskPrice              683323 non-null  float64
 8   BidImpliedVolatility  683323 non-null  float64
 9   AskImpliedVolatility  683323 non-null  float64
 10  OpenInterest          683323 non-null  float64
 11  Volume                683323 non-null  float64
 12  Delta                 683323 non-null  float64
 13  Gamma                 683323 non-null  float64
 14  Vega                  683323 non-null  float64
 15  

In [28]:
def compute_px(flag,adjspot,days_to_expiry,risk_free_rate,stk,imp_vol):
        days =365
        
        
        px = bs(flag,adjspot,stk,days_to_expiry/days,risk_free_rate,imp_vol)
        valid_px = .0004
        if px<=valid_px:
                px = valid_px
        if days_to_expiry == 0:
                d=g=v=th=rh = 0
        else:
        #print(px)
                d = delta(flag,adjspot,stk,days_to_expiry/days,risk_free_rate,imp_vol)
                g = gamma(flag,adjspot,stk,days_to_expiry/days,risk_free_rate,imp_vol)
                v = vega(flag,adjspot,stk,days_to_expiry/days,risk_free_rate,imp_vol)
                th = theta(flag,adjspot,stk,days_to_expiry/days,risk_free_rate,imp_vol)
                rh = rho(flag,adjspot,stk,days_to_expiry/days,risk_free_rate,imp_vol)
        
        return (px, d,g,v,th,rh)
        

In [23]:
df[df.LastTradePrice>0].LastTradePrice.min()

0.0004

In [ ]:
def compute_iv(x):
    try:
        implied_vol = iv.implied_volatility(x.AskPrice,x.AdjSpot,x.AdjStrike,x.days_to_expiry/365,x.risk_free_rate,x.CallPut)
    except Exception as e:
        # check exception 
        implied_vol = 0
    implied_vol = min(20,implied_vol)
    return implied_vol

In [11]:
df.columns

Index(['Ticker', 'TradeDate', 'Strike', 'ExpiryDate', 'CallPut',
       'LastTradePrice', 'BidPrice', 'AskPrice', 'BidImpliedVolatility',
       'AskImpliedVolatility', 'OpenInterest', 'Volume', 'Delta', 'Gamma',
       'Vega', 'Theta', 'Rho', 'AdjStrike', 'Symbol', 'Spot', 'AdjSpot',
       'days_to_expiry', 'Month_To_Expiry', 'AdjExpiry', 'ExpirySpot',
       'risk_free_rate', 'AbsMoneyness', 'Moneyness', 'ImpliedVolatility'],
      dtype='object')

In [ ]:
df['ImpliedVolatility']=df.apply(lambda x:compute_px(x),axis=1)

c:\Users\shyam\miniconda3\envs\SPX\Lib\site-packages\py_lets_be_rational\lets_be_rational.py:612: RuntimeWarning: divide by zero encountered in scalar divide
  return _unchecked_normalised_implied_volatility_from_a_transformed_rational_guess_with_limited_iterations(


In [29]:
df['params'] = df.apply(lambda x:compute_px(x.CallPut,x.AdjSpot,x.days_to_expiry,x.risk_free_rate,x.AdjStrike,x.ImpliedVolatility),axis=1)

c:\Users\shyam\miniconda3\envs\SPX\Lib\site-packages\py_vollib\ref_python\black_scholes\__init__.py:87: RuntimeWarning: divide by zero encountered in scalar divide
  return numerator / denominator
c:\Users\shyam\miniconda3\envs\SPX\Lib\site-packages\py_vollib\black_scholes\greeks\analytical.py:190: RuntimeWarning: invalid value encountered in scalar divide
  return pdf(d_1)/(S*sigma*numpy.sqrt(t))


In [33]:
for i,item in enumerate(['px', 'd','g','v','th','rh']):
    df[item] = df.params.apply(lambda x:x[i])

In [45]:
df[['days_to_expiry','px','AskPrice','Delta','d','Vega','v','Gamma','g','Theta','th','Rho','rh']].iloc[10000:10010]

,days_to_expiry,px,AskPrice,Delta,d,Vega,v,Gamma,g,Theta,th,Rho,rh
10000,93,1.4286,1.4286,-0.7003,-0.701736,0.5303,0.019282,0.0071,0.197538,-0.0897,-0.003235,-65.2510,-0.023335
10001,93,1.7036,1.7036,-0.7620,-0.763617,0.5027,0.017142,0.0064,0.176974,-0.0788,-0.002846,-72.0668,-0.025773
10002,93,0.0343,0.0343,0.0449,0.048620,0.1558,0.005608,0.0019,0.056044,-0.0252,-0.000978,0.0330,0.001277
10003,93,0.0257,0.0257,0.0325,0.037406,0.1088,0.004537,0.0015,0.044781,-0.0194,-0.000801,0.0240,0.000984
10004,93,0.0196,0.0196,0.0232,0.029113,0.0902,0.003690,0.0011,0.035905,-0.0147,-0.000661,0.0172,0.000767
10005,93,0.0154,0.0154,0.0164,0.023159,0.0592,0.003047,0.0008,0.029131,-0.0109,-0.000555,0.0121,0.000611
10006,93,0.0121,0.0121,0.0115,0.018422,0.0474,0.002510,0.0006,0.023616,-0.0081,-0.000465,0.0086,0.000486
10007,93,0.0332,0.0332,-0.0289,-0.029188,0.0490,0.003698,0.0005,0.024656,-0.0267,-0.000963,-2.5081,-0.000904
10008,93,0.0432,0.0432,-0.0383,-0.038461,0.0760,0.004641,0.0008,0.032540,-0.0320,-0.001149,-3.3124,-0.001190
10009,93,0.0575,0.0575,-0.0509,-0.051307,0.1129,0.005854,0.0012,0.042978,-0.0384,-0.001384,-4.4050,-0.001587


In [60]:
df[df.g == df.g.max()][['AdjSpot','AdjStrike','days_to_expiry','px','AskPrice','Delta','d','Vega','v','Gamma','g','Theta','th','Rho','rh','CallPut']]

,AdjSpot,AdjStrike,days_to_expiry,px,AskPrice,Delta,d,Vega,v,Gamma,g,Theta,th,Rho,rh,CallPut
6510,8.9243,8.9286,2,0.0393,0.0393,-0.4952,-0.516001,0.074,0.002633,0.1495,4.281552,-0.2328,-0.009258,-0.6843,-0.000254,p


In [58]:
df.g.max()

4.281552371760371

In [4]:
base_path = os.path.join(os.getcwd(),'Data','AAPL')
base_path

'c:\\Users\\shyam\\Dp Lrn\\Options\\Data\\AAPL'

In [5]:
df = pd.read_csv(os.path.join(base_path,'Options_AAPL_Monthly.csv'))

In [5]:
df[df.TradeDate==20100129]

,Ticker,TradeDate,Strike,ExpiryDate,CallPut,LastTradePrice,BidPrice,AskPrice,BidImpliedVolatility,AskImpliedVolatility,OpenInterest,Volume,Delta,Gamma,Vega,Theta,Rho
453234,AAPL,20100129,290.0,20100320,p,97.94,96.70,98.35,0.0,0.5619,1.0,0.0,-1.0000,0.0002,0.0043,0.0000,-39.7248
453235,AAPL,20100129,300.0,20100320,p,107.94,106.65,108.35,0.0,0.5996,0.0,0.0,-1.0000,0.0001,0.0017,0.0000,-41.0949
453236,AAPL,20100129,90.0,20100417,c,102.09,101.35,103.60,0.0,1.0420,127.0,0.0,0.9993,0.0000,0.0022,-0.0009,0.1920
453237,AAPL,20100129,95.0,20100417,c,97.10,96.35,98.65,0.0,0.9833,244.0,0.0,0.9985,0.0001,0.0042,-0.0017,0.2024
453238,AAPL,20100129,100.0,20100417,c,92.12,91.40,93.65,0.0,0.9212,192.0,0.0,0.9972,0.0002,0.0075,-0.0029,0.2124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480385,AAPL,20100129,240.0,20100320,p,48.57,47.60,48.55,0.0,0.3691,292.0,17.0,-0.9994,0.0045,0.0817,-0.0003,-32.8607
480386,AAPL,20100129,250.0,20100320,p,58.24,57.35,58.25,0.0,0.3743,233.0,3.0,-0.9999,0.0027,0.0484,0.0000,-34.2445
480387,AAPL,20100129,260.0,20100320,p,68.07,67.20,68.15,0.0,0.3958,20.0,2.0,-1.0000,0.0015,0.0317,0.0000,-35.6159
480388,AAPL,20100129,270.0,20100320,p,77.99,76.70,78.20,0.0,0.4495,134.0,0.0,-1.0000,0.0008,0.0166,0.0000,-36.9856


In [5]:
df.head()

,Ticker,TradeDate,Strike,ExpiryDate,CallPut,LastTradePrice,BidPrice,AskPrice,BidImpliedVolatility,AskImpliedVolatility,OpenInterest,Volume,Delta,Gamma,Vega,Theta,Rho
0,AAPL,20181018,225.0,20181026,p,9.71,9.55,9.90,0.2741,0.3150,3664.0,783.0,-0.8158,0.0286,0.0842,-0.1565,-4.0791
1,AAPL,20181018,222.5,20181026,p,7.77,7.70,7.90,0.2902,0.3092,2909.0,560.0,-0.7360,0.0346,0.1059,-0.1957,-3.6591
2,AAPL,20181018,220.0,20181026,p,6.04,6.05,6.15,0.3010,0.3092,5442.0,4612.0,-0.6439,0.0390,0.1219,-0.2272,-3.1855
3,AAPL,20181018,217.5,20181026,p,4.58,4.60,4.70,0.3068,0.3147,2758.0,5465.0,-0.5450,0.0407,0.1270,-0.2461,-2.6848
4,AAPL,20181018,215.0,20181026,p,3.41,3.50,3.55,0.3208,0.3248,5395.0,10621.0,-0.4464,0.0395,0.1259,-0.2551,-2.1927


In [88]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 683323 entries, 0 to 683322
Data columns (total 41 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   Ticker                      683323 non-null  object        
 1   TradeDate                   683323 non-null  int64         
 2   ExpiryDate                  683323 non-null  int64         
 3   CallPut                     683323 non-null  object        
 4   AdjStrike                   683323 non-null  float64       
 5   Strike                      683323 non-null  float64       
 6   AdjSpot                     683323 non-null  float64       
 7   Spot                        683323 non-null  float64       
 8   Month_To_Expiry             683323 non-null  int32         
 9   days_to_expiry              683323 non-null  int64         
 10  ExpirySpot                  683323 non-null  float64       
 11  AdjExpiry                   683323 non-

In [7]:
df_stock = pd.read_csv(os.path.join(base_path,'AAPL.CSV'))

In [8]:
df_stock = df_stock.drop(columns=[' 0.21000000', ' 0.21000000.1', ' 0.20000000',
                                  ' 0.21000000.2', ' 2191845', '       0', ' 2191845.1', '       0.1',
                                  ' 0.21000000.3', ' 0.21000000.4', 'Technology', '              0',
                                  '  -99999999.000', '  -99999999.000.1', '  -99999999.000.2',
                                  '          0.000', '         460287'])
df_stock.head()

,19980501,28.00000000
0,19980504,29.0625
1,19980505,29.6875
2,19980506,30.3125
3,19980507,30.1875
4,19980508,30.4375


In [9]:
df_stock.columns

Index(['19980501', '28.00000000'], dtype='object')

In [10]:
df_stock = df_stock.rename(columns={'19980501': 'TradeDate',"28.00000000": 'Spot'})

In [11]:
df_stock.head()

,TradeDate,Spot
0,19980504,29.0625
1,19980505,29.6875
2,19980506,30.3125
3,19980507,30.1875
4,19980508,30.4375


In [12]:
def adjust_stock(x,sp = 'Spot',rf = 4):
    
    if x.TradeDate<20140609:
        factor = 28
    elif 20140609<=x.TradeDate<20200831:
        factor = 4
    else:
        factor = 1
    {''}
    adj_spot =  round(x[sp]/factor,rf)
    return adj_spot


In [13]:
df_stock['AdjSpot'] =  df_stock.apply(lambda x:adjust_stock(x,sp="Spot"),axis=1)

In [14]:
df = df.merge(df_stock, how='left', left_on='TradeDate', right_on='TradeDate')
df.columns

Index(['Ticker', 'TradeDate', 'Strike', 'ExpiryDate', 'CallPut',
       'LastTradePrice', 'BidPrice', 'AskPrice', 'BidImpliedVolatility',
       'AskImpliedVolatility', 'OpenInterest', 'Volume', 'Delta', 'Gamma',
       'Vega', 'Theta', 'Rho', 'Spot', 'AdjSpot'],
      dtype='object')

In [15]:
df = df.sort_values("TradeDate", ascending=True)

In [16]:
df.columns

Index(['Ticker', 'TradeDate', 'Strike', 'ExpiryDate', 'CallPut',
       'LastTradePrice', 'BidPrice', 'AskPrice', 'BidImpliedVolatility',
       'AskImpliedVolatility', 'OpenInterest', 'Volume', 'Delta', 'Gamma',
       'Vega', 'Theta', 'Rho', 'Spot', 'AdjSpot'],
      dtype='object')

In [17]:
df.head()

,Ticker,TradeDate,Strike,ExpiryDate,CallPut,LastTradePrice,BidPrice,AskPrice,BidImpliedVolatility,AskImpliedVolatility,OpenInterest,Volume,Delta,Gamma,Vega,Theta,Rho,Spot,AdjSpot
453545,AAPL,20100121,290.0,20100717,p,84.00,82.45,84.00,0.2646,0.3438,201.0,0.0,-0.9280,0.0036,0.2747,-0.0171,-134.0003,208.07,7.4311
453544,AAPL,20100121,280.0,20100717,p,74.86,73.20,75.15,0.2845,0.3562,331.0,0.0,-0.8886,0.0043,0.3073,-0.0249,-125.6507,208.07,7.4311
453543,AAPL,20100121,270.0,20100717,p,65.70,65.25,65.75,0.3371,0.3509,504.0,0.0,-0.8317,0.0051,0.3818,-0.0338,-115.8790,208.07,7.4311
453574,AAPL,20100121,130.0,20110122,c,83.96,83.10,84.75,0.4075,0.4553,2395.0,5.0,0.9070,0.0018,0.3618,-0.0231,1.0505,208.07,7.4311
453541,AAPL,20100121,250.0,20100717,p,48.98,48.60,48.80,0.3460,0.3502,432.0,13.0,-0.7354,0.0065,0.4789,-0.0453,-97.9959,208.07,7.4311


In [18]:
df['AdjStrike'] = df.apply(lambda x:adjust_stock(x,sp="Strike"),axis=1)


In [19]:
df['AbsMoneyness'] = 100*(df.AdjStrike/df.AdjSpot)
df['Moneyness'] =(df.AbsMoneyness /10).round()*10

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 683323 entries, 453545 to 520303
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Ticker                683323 non-null  object 
 1   TradeDate             683323 non-null  int64  
 2   Strike                683323 non-null  float64
 3   ExpiryDate            683323 non-null  int64  
 4   CallPut               683323 non-null  object 
 5   LastTradePrice        683323 non-null  float64
 6   BidPrice              683323 non-null  float64
 7   AskPrice              683323 non-null  float64
 8   BidImpliedVolatility  683323 non-null  float64
 9   AskImpliedVolatility  683323 non-null  float64
 10  OpenInterest          683323 non-null  float64
 11  Volume                683323 non-null  float64
 12  Delta                 683323 non-null  float64
 13  Gamma                 683323 non-null  float64
 14  Vega                  683323 non-null  float64
 15  

In [21]:
df["FmtTradeDate"] = pd.to_datetime(df["TradeDate"], format="%Y%m%d")
df["FmtExpiryDate"] = pd.to_datetime(df["ExpiryDate"], format="%Y%m%d")
df["days_to_expiry"] = (df["FmtExpiryDate"] - df["FmtTradeDate"]).dt.days

In [22]:
df_stock["FmtTradeDate"] = pd.to_datetime(df_stock["TradeDate"], format="%Y%m%d")

In [23]:

df['AdjExpiry'] = df['FmtExpiryDate']
df.loc[df['AdjExpiry'].dt.weekday == 5, 'AdjExpiry'] -= pd.Timedelta(days=1)

In [24]:
df.loc[df.ExpiryDate==20140419,'AdjExpiry']-=pd.Timedelta(days=1)

In [25]:
df_stock.head()

,TradeDate,Spot,AdjSpot,FmtTradeDate
0,19980504,29.0625,1.0379,1998-05-04
1,19980505,29.6875,1.0603,1998-05-05
2,19980506,30.3125,1.0826,1998-05-06
3,19980507,30.1875,1.0781,1998-05-07
4,19980508,30.4375,1.0871,1998-05-08


In [26]:
df = df.merge(df_stock, how='left', left_on='AdjExpiry', right_on='FmtTradeDate',suffixes=('', '_expiry')
              ).drop(columns=['FmtTradeDate_expiry','TradeDate_expiry','Spot_expiry']).rename(columns={'AdjSpot_expiry': 'ExpirySpot'})

In [27]:
df[(df.TradeDate==20130214)&(df.AdjExpiry=='2013/07/19')&(df.CallPut=='c')&(df.Volume>0)][['TradeDate','CallPut','AdjSpot','AdjStrike','OpenInterest','Volume','AskPrice','BidPrice','LastTradePrice']].sort_values(by='AdjStrike').head(50)

,TradeDate,CallPut,AdjSpot,AdjStrike,OpenInterest,Volume,AskPrice,BidPrice,LastTradePrice
102912,20130214,c,16.6639,9.6429,0.0,1.0,198.00,196.40,194.32
102919,20130214,c,16.6639,13.2143,53.0,5.0,101.55,100.85,100.49
103119,20130214,c,16.6639,13.5714,13.0,1.0,92.65,92.10,92.00
103085,20130214,c,16.6639,13.9286,46.0,5.0,84.35,83.35,84.55
103098,20130214,c,16.6639,14.2857,406.0,2.0,76.05,75.50,76.48
103074,20130214,c,16.6639,14.6429,129.0,3.0,68.20,67.75,68.67
103069,20130214,c,16.6639,15.0000,210.0,22.0,60.90,60.40,61.17
103054,20130214,c,16.6639,15.1786,252.0,2.0,57.45,56.90,57.56
103055,20130214,c,16.6639,15.3571,103.0,6.0,54.00,53.50,54.00
103057,20130214,c,16.6639,15.7143,465.0,27.0,47.45,47.10,47.26


In [28]:
df['AskPrice'] = df.apply(lambda x:adjust_stock(x,sp="AskPrice"),axis=1)
df['BidPrice'] = df.apply(lambda x:adjust_stock(x,sp="BidPrice"),axis=1)
df['LastTradePrice'] = df.apply(lambda x:adjust_stock(x,sp="LastTradePrice"),axis=1)

In [29]:
df[(df.TradeDate==20130214)&(df.AdjExpiry=='2013/07/19')&(df.CallPut=='c')&(df.Volume>0)][['TradeDate','CallPut','AdjSpot','AdjStrike','OpenInterest','Volume','AskPrice','BidPrice','LastTradePrice']].sort_values(by='AdjStrike').head()

,TradeDate,CallPut,AdjSpot,AdjStrike,OpenInterest,Volume,AskPrice,BidPrice,LastTradePrice
102912,20130214,c,16.6639,9.6429,0.0,1.0,7.0714,7.0143,6.9400
102919,20130214,c,16.6639,13.2143,53.0,5.0,3.6268,3.6018,3.5889
103119,20130214,c,16.6639,13.5714,13.0,1.0,3.3089,3.2893,3.2857
103085,20130214,c,16.6639,13.9286,46.0,5.0,3.0125,2.9768,3.0196
103098,20130214,c,16.6639,14.2857,406.0,2.0,2.7161,2.6964,2.7314


In [30]:
df.columns

Index(['Ticker', 'TradeDate', 'Strike', 'ExpiryDate', 'CallPut',
       'LastTradePrice', 'BidPrice', 'AskPrice', 'BidImpliedVolatility',
       'AskImpliedVolatility', 'OpenInterest', 'Volume', 'Delta', 'Gamma',
       'Vega', 'Theta', 'Rho', 'Spot', 'AdjSpot', 'AdjStrike', 'AbsMoneyness',
       'Moneyness', 'FmtTradeDate', 'FmtExpiryDate', 'days_to_expiry',
       'AdjExpiry', 'ExpirySpot'],
      dtype='object')

In [31]:
df[(df.ExpirySpot.isnull())&(df.FmtExpiryDate.dt.day_name() =='Friday')].ExpiryDate.value_counts()

ExpiryDate
20261218    5190
20270115    3970
20271217    3348
20260918    2102
20260320    1928
20260515    1592
20260220    1518
20260417    1004
20280121     880
20260821     846
20260717     364
20260130     232
20280317     224
20260206     116
20260213      58
Name: count, dtype: int64

In [32]:
df.tail()

,Ticker,TradeDate,Strike,ExpiryDate,CallPut,LastTradePrice,BidPrice,AskPrice,BidImpliedVolatility,AskImpliedVolatility,...,Spot,AdjSpot,AdjStrike,AbsMoneyness,Moneyness,FmtTradeDate,FmtExpiryDate,days_to_expiry,AdjExpiry,ExpirySpot
683318,AAPL,20251231,145.0,20260123,p,0.0,0.0,0.23,0.0,1.1226,...,271.86,271.86,145.0,53.336276,50.0,2025-12-31,2026-01-23,23,2026-01-23,248.04
683319,AAPL,20251231,150.0,20260123,p,0.0,0.0,0.23,0.0,1.0679,...,271.86,271.86,150.0,55.175458,60.0,2025-12-31,2026-01-23,23,2026-01-23,248.04
683320,AAPL,20251231,155.0,20260123,p,0.0,0.0,0.23,0.0,1.0151,...,271.86,271.86,155.0,57.014640,60.0,2025-12-31,2026-01-23,23,2026-01-23,248.04
683321,AAPL,20251231,160.0,20260123,p,0.0,0.0,0.23,0.0,0.9640,...,271.86,271.86,160.0,58.853822,60.0,2025-12-31,2026-01-23,23,2026-01-23,248.04
683322,AAPL,20251231,165.0,20260123,p,0.0,0.0,0.23,0.0,0.9135,...,271.86,271.86,165.0,60.693004,60.0,2025-12-31,2026-01-23,23,2026-01-23,248.04


In [33]:
df[df.ExpirySpot.isnull()].FmtExpiryDate.dt.day_name().value_counts()

FmtExpiryDate
Friday      23372
Thursday     8980
Name: count, dtype: int64

In [34]:
df.iloc[-1].Spot

np.float64(271.86)

In [35]:
df['ExpirySpot'] = df['ExpirySpot'].fillna(df.iloc[-1]['Spot'])

In [36]:
df[df.ExpirySpot.isnull()].FmtExpiryDate.dt.day_name().value_counts()

Series([], Name: count, dtype: int64)

In [37]:
np.sort(df.FmtTradeDate.unique())

array(['2010-01-21T00:00:00.000000000', '2010-01-29T00:00:00.000000000',
       '2010-02-18T00:00:00.000000000', '2010-02-26T00:00:00.000000000',
       '2010-03-18T00:00:00.000000000', '2010-03-31T00:00:00.000000000',
       '2010-04-22T00:00:00.000000000', '2010-04-30T00:00:00.000000000',
       '2010-05-20T00:00:00.000000000', '2010-05-28T00:00:00.000000000',
       '2010-06-17T00:00:00.000000000', '2010-06-30T00:00:00.000000000',
       '2010-07-15T00:00:00.000000000', '2010-07-30T00:00:00.000000000',
       '2010-08-19T00:00:00.000000000', '2010-08-31T00:00:00.000000000',
       '2010-09-16T00:00:00.000000000', '2010-09-30T00:00:00.000000000',
       '2010-10-14T00:00:00.000000000', '2010-10-29T00:00:00.000000000',
       '2010-11-18T00:00:00.000000000', '2010-11-30T00:00:00.000000000',
       '2010-12-16T00:00:00.000000000', '2010-12-31T00:00:00.000000000',
       '2011-01-20T00:00:00.000000000', '2011-01-31T00:00:00.000000000',
       '2011-02-17T00:00:00.000000000', '2011-02-28

In [38]:
for year in df.FmtExpiryDate.dt.year.unique():

    print(f'year {year} has  {len(df[df.AdjExpiry.dt.year==year].AdjExpiry.unique())} expiry dates')

year 2010 has  27 expiry dates
year 2011 has  42 expiry dates
year 2012 has  42 expiry dates
year 2013 has  52 expiry dates
year 2014 has  52 expiry dates
year 2015 has  53 expiry dates
year 2016 has  52 expiry dates
year 2017 has  52 expiry dates
year 2018 has  52 expiry dates
year 2019 has  52 expiry dates
year 2020 has  53 expiry dates
year 2021 has  52 expiry dates
year 2022 has  52 expiry dates
year 2023 has  52 expiry dates
year 2024 has  52 expiry dates
year 2025 has  52 expiry dates
year 2026 has  16 expiry dates
year 2027 has  3 expiry dates
year 2028 has  2 expiry dates


In [39]:
np.sort(df[df.AdjExpiry.dt.year==2011].AdjExpiry.dt.date.unique())

array([datetime.date(2011, 1, 7), datetime.date(2011, 1, 21),
       datetime.date(2011, 1, 28), datetime.date(2011, 2, 4),
       datetime.date(2011, 2, 18), datetime.date(2011, 2, 25),
       datetime.date(2011, 3, 4), datetime.date(2011, 3, 18),
       datetime.date(2011, 3, 25), datetime.date(2011, 4, 1),
       datetime.date(2011, 4, 8), datetime.date(2011, 4, 15),
       datetime.date(2011, 4, 21), datetime.date(2011, 4, 29),
       datetime.date(2011, 5, 6), datetime.date(2011, 5, 20),
       datetime.date(2011, 5, 27), datetime.date(2011, 6, 3),
       datetime.date(2011, 6, 17), datetime.date(2011, 6, 24),
       datetime.date(2011, 7, 1), datetime.date(2011, 7, 8),
       datetime.date(2011, 7, 15), datetime.date(2011, 7, 22),
       datetime.date(2011, 7, 29), datetime.date(2011, 8, 5),
       datetime.date(2011, 8, 19), datetime.date(2011, 8, 26),
       datetime.date(2011, 9, 2), datetime.date(2011, 9, 16),
       datetime.date(2011, 9, 23), datetime.date(2011, 9, 30),
   

In [40]:
last_dates = (
    df.groupby(df["FmtTradeDate"].dt.to_period("M"))["FmtTradeDate"]
      .max()
)
last_dates

FmtTradeDate
2010-01   2010-01-29
2010-02   2010-02-26
2010-03   2010-03-31
2010-04   2010-04-30
2010-05   2010-05-28
             ...    
2025-08   2025-08-29
2025-09   2025-09-30
2025-10   2025-10-31
2025-11   2025-11-28
2025-12   2025-12-31
Freq: M, Name: FmtTradeDate, Length: 192, dtype: datetime64[ns]

In [43]:
def symbol(date):
    if date in last_dates.values:
        return 'EV'
    else:
        return 'SP'
    


In [44]:
df['Symbol'] = df.FmtTradeDate.apply(lambda x: symbol(x))

In [45]:
df.Symbol.value_counts()

Symbol
SP    343016
EV    340307
Name: count, dtype: int64

In [46]:
df.head()

,Ticker,TradeDate,Strike,ExpiryDate,CallPut,LastTradePrice,BidPrice,AskPrice,BidImpliedVolatility,AskImpliedVolatility,...,AdjSpot,AdjStrike,AbsMoneyness,Moneyness,FmtTradeDate,FmtExpiryDate,days_to_expiry,AdjExpiry,ExpirySpot,Symbol
0,AAPL,20100121,290.0,20100717,p,3.0000,2.9446,3.0000,0.2646,0.3438,...,7.4311,10.3571,139.375059,140.0,2010-01-21,2010-07-17,177,2010-07-16,8.9250,SP
1,AAPL,20100121,280.0,20100717,p,2.6736,2.6143,2.6839,0.2845,0.3562,...,7.4311,10.0000,134.569579,130.0,2010-01-21,2010-07-17,177,2010-07-16,8.9250,SP
2,AAPL,20100121,270.0,20100717,p,2.3464,2.3304,2.3482,0.3371,0.3509,...,7.4311,9.6429,129.764100,130.0,2010-01-21,2010-07-17,177,2010-07-16,8.9250,SP
3,AAPL,20100121,130.0,20110122,c,2.9986,2.9679,3.0268,0.4075,0.4553,...,7.4311,4.6429,62.479310,60.0,2010-01-21,2011-01-22,366,2011-01-21,11.6686,SP
4,AAPL,20100121,250.0,20100717,p,1.7493,1.7357,1.7429,0.3460,0.3502,...,7.4311,8.9286,120.151794,120.0,2010-01-21,2010-07-17,177,2010-07-16,8.9250,SP


In [47]:

df['Month_To_Expiry'] = (df.FmtExpiryDate.dt.year - df.FmtTradeDate.dt.year) * 12 + (df.FmtExpiryDate.dt.month - df.FmtTradeDate.dt.month)


In [48]:
df.shape

(683323, 29)

In [49]:
df.head()

,Ticker,TradeDate,Strike,ExpiryDate,CallPut,LastTradePrice,BidPrice,AskPrice,BidImpliedVolatility,AskImpliedVolatility,...,AdjStrike,AbsMoneyness,Moneyness,FmtTradeDate,FmtExpiryDate,days_to_expiry,AdjExpiry,ExpirySpot,Symbol,Month_To_Expiry
0,AAPL,20100121,290.0,20100717,p,3.0000,2.9446,3.0000,0.2646,0.3438,...,10.3571,139.375059,140.0,2010-01-21,2010-07-17,177,2010-07-16,8.9250,SP,6
1,AAPL,20100121,280.0,20100717,p,2.6736,2.6143,2.6839,0.2845,0.3562,...,10.0000,134.569579,130.0,2010-01-21,2010-07-17,177,2010-07-16,8.9250,SP,6
2,AAPL,20100121,270.0,20100717,p,2.3464,2.3304,2.3482,0.3371,0.3509,...,9.6429,129.764100,130.0,2010-01-21,2010-07-17,177,2010-07-16,8.9250,SP,6
3,AAPL,20100121,130.0,20110122,c,2.9986,2.9679,3.0268,0.4075,0.4553,...,4.6429,62.479310,60.0,2010-01-21,2011-01-22,366,2011-01-21,11.6686,SP,12
4,AAPL,20100121,250.0,20100717,p,1.7493,1.7357,1.7429,0.3460,0.3502,...,8.9286,120.151794,120.0,2010-01-21,2010-07-17,177,2010-07-16,8.9250,SP,6


In [50]:
df[(df.CallPut =="c") & (df.Moneyness == 90)&(df.Month_To_Expiry==3)&(df.Symbol == 'EV')].TradeDate.unique()

array([20100129, 20100430, 20100730, 20101029, 20101130, 20101231,
       20110131, 20110228, 20110331, 20110429, 20110531, 20110630,
       20110729, 20110831, 20110930, 20111031, 20111130, 20111230,
       20120131, 20120229, 20120330, 20120430, 20120531, 20120629,
       20120731, 20120831, 20120928, 20121031, 20121130, 20121231,
       20130131, 20130228, 20130328, 20130430, 20130531, 20130628,
       20130731, 20130830, 20130930, 20131031, 20131129, 20131231,
       20140131, 20140228, 20140331, 20140430, 20140530, 20140630,
       20140731, 20140829, 20140930, 20141031, 20141128, 20141231,
       20150130, 20150227, 20150331, 20150430, 20150529, 20150630,
       20150731, 20150831, 20150930, 20151030, 20151130, 20151231,
       20160129, 20160229, 20160331, 20160429, 20160531, 20160630,
       20160729, 20160831, 20160930, 20161031, 20161130, 20161230,
       20170131, 20170228, 20170331, 20170428, 20170531, 20170630,
       20170731, 20170831, 20170929, 20171031, 20171130, 20180

In [51]:
np.sort(df.Moneyness.unique())

array([  0.,  10.,  20.,  30.,  40.,  50.,  60.,  70.,  80.,  90., 100.,
       110., 120., 130., 140., 150., 160., 170., 180., 190., 200., 210.,
       220., 230., 240., 250., 260., 270.])

In [8]:
df_rates = pd.read_csv(os.path.join(base_path,'bill-rates-2002-2023.csv'))


In [9]:
df_rates_24 = pd.read_csv(os.path.join(base_path,'daily-treasury-rates-2024.csv'))
df_rates_25 = pd.read_csv(os.path.join(base_path,'daily-treasury-rates-2025.csv'))
df_rates_26 = pd.read_csv(os.path.join(base_path,'daily-treasury-rates-2026.csv'))

In [5]:
df_rates_25= df_rates_25[list(df_rates_24.columns)]
df_rates_26= df_rates_26[list(df_rates_24.columns)]

In [10]:
cols = list(df_rates_25.columns)
cols = [i.lower() for i in cols]
cols

['date',
 '4 weeks bank discount',
 '4 weeks coupon equivalent',
 '6 weeks bank discount',
 '6 weeks coupon equivalent',
 '8 weeks bank discount',
 '8 weeks coupon equivalent',
 '13 weeks bank discount',
 '13 weeks coupon equivalent',
 '17 weeks bank discount',
 '17 weeks coupon equivalent',
 '26 weeks bank discount',
 '26 weeks coupon equivalent',
 '52 weeks bank discount',
 '52 weeks coupon equivalent']

In [11]:
cols_dict = {}
for i in range(len(cols)):
    cols_dict[df_rates_25.columns[i]] = cols[i]

cols_dict

{'Date': 'date',
 '4 WEEKS BANK DISCOUNT': '4 weeks bank discount',
 '4 WEEKS COUPON EQUIVALENT': '4 weeks coupon equivalent',
 '6 WEEKS BANK DISCOUNT': '6 weeks bank discount',
 '6 WEEKS COUPON EQUIVALENT': '6 weeks coupon equivalent',
 '8 WEEKS BANK DISCOUNT': '8 weeks bank discount',
 '8 WEEKS COUPON EQUIVALENT': '8 weeks coupon equivalent',
 '13 WEEKS BANK DISCOUNT': '13 weeks bank discount',
 '13 WEEKS COUPON EQUIVALENT': '13 weeks coupon equivalent',
 '17 WEEKS BANK DISCOUNT': '17 weeks bank discount',
 '17 WEEKS COUPON EQUIVALENT': '17 weeks coupon equivalent',
 '26 WEEKS BANK DISCOUNT': '26 weeks bank discount',
 '26 WEEKS COUPON EQUIVALENT': '26 weeks coupon equivalent',
 '52 WEEKS BANK DISCOUNT': '52 weeks bank discount',
 '52 WEEKS COUPON EQUIVALENT': '52 weeks coupon equivalent'}

In [90]:
A= [1,2,3,4]
for i,j in enumerate(A):
    A[i]=j+1
A

[2, 3, 4, 5]

In [12]:
df_rates_24 =  df_rates_24.rename(columns=cols_dict)
df_rates_25 =  df_rates_25.rename(columns=cols_dict)
df_rates_26 =  df_rates_26.rename(columns=cols_dict)

In [13]:
df_rates_25

,date,4 weeks bank discount,4 weeks coupon equivalent,6 weeks bank discount,6 weeks coupon equivalent,8 weeks bank discount,8 weeks coupon equivalent,13 weeks bank discount,13 weeks coupon equivalent,17 weeks bank discount,17 weeks coupon equivalent,26 weeks bank discount,26 weeks coupon equivalent,52 weeks bank discount,52 weeks coupon equivalent
0,12/31/2025,3.59,3.65,3.61,3.68,3.58,3.65,3.57,3.65,3.55,3.64,3.50,3.61,3.35,3.48
1,12/30/2025,3.50,3.56,3.57,3.63,3.51,3.58,3.55,3.63,3.53,3.62,3.50,3.61,3.34,3.47
2,12/29/2025,3.54,3.60,3.55,3.61,3.56,3.63,3.57,3.65,3.56,3.65,3.50,3.61,3.35,3.48
3,12/26/2025,3.56,3.62,3.53,3.59,3.58,3.65,3.56,3.64,3.56,3.65,3.48,3.59,3.35,3.48
4,12/24/2025,3.58,3.64,3.58,3.64,3.59,3.66,3.60,3.68,3.56,3.65,3.48,3.59,3.36,3.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
244,01-08-2025,4.24,4.31,NaN,NaN,4.23,4.32,4.21,4.31,4.19,4.31,4.13,4.28,4.00,4.18
245,01-07-2025,4.24,4.31,NaN,NaN,4.23,4.32,4.21,4.31,4.20,4.32,4.13,4.28,4.01,4.19
246,01-06-2025,4.24,4.31,NaN,NaN,4.25,4.34,4.21,4.31,4.20,4.32,4.12,4.27,3.99,4.17
247,01-03-2025,4.25,4.32,NaN,NaN,4.24,4.33,4.20,4.30,4.19,4.31,4.14,4.29,4.00,4.18


In [14]:
df_rates_25.columns

Index(['date', '4 weeks bank discount', '4 weeks coupon equivalent',
       '6 weeks bank discount', '6 weeks coupon equivalent',
       '8 weeks bank discount', '8 weeks coupon equivalent',
       '13 weeks bank discount', '13 weeks coupon equivalent',
       '17 weeks bank discount', '17 weeks coupon equivalent',
       '26 weeks bank discount', '26 weeks coupon equivalent',
       '52 weeks bank discount', '52 weeks coupon equivalent'],
      dtype='object')

In [15]:
df_rates = pd.concat([df_rates, df_rates_24,df_rates_25,df_rates_26], ignore_index=True)

In [ ]:
df_rates = df_rates.drop(columns=['4 weeks bank discount','6 weeks bank discount','8 weeks bank discount', 
                       '13 weeks bank discount','17 weeks bank discount',
                       '26 weeks bank discount','52 weeks bank discount'])


In [17]:
df_rates.head()

,date,4 weeks coupon equivalent,8 weeks coupon equivalent,13 weeks coupon equivalent,17 weeks coupon equivalent,26 weeks coupon equivalent,52 weeks coupon equivalent,6 weeks bank discount,6 weeks coupon equivalent
0,01/02/2002,1.74,NaN,1.74,NaN,1.85,NaN,NaN,NaN
1,01/03/2002,1.73,NaN,1.73,NaN,1.82,NaN,NaN,NaN
2,01/04/2002,1.72,NaN,1.72,NaN,1.82,NaN,NaN,NaN
3,01/07/2002,1.71,NaN,1.69,NaN,1.77,NaN,NaN,NaN
4,01/08/2002,1.69,NaN,1.68,NaN,1.78,NaN,NaN,NaN


In [18]:
df_rates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6023 entries, 0 to 6022
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   date                        6023 non-null   object 
 1   4 weeks coupon equivalent   6021 non-null   float64
 2   8 weeks coupon equivalent   1820 non-null   float64
 3   13 weeks coupon equivalent  6022 non-null   float64
 4   17 weeks coupon equivalent  818 non-null    float64
 5   26 weeks coupon equivalent  6022 non-null   float64
 6   52 weeks coupon equivalent  4421 non-null   float64
 7   6 weeks bank discount       237 non-null    float64
 8   6 weeks coupon equivalent   237 non-null    float64
dtypes: float64(8), object(1)
memory usage: 423.6+ KB


In [64]:
def date_formatting(date):
    obj = date.split('/')
    try:
        if len(obj[0])==1:
            obj[0] = f'0{obj[0]}'
        if len(obj[1])==1:
            obj[1] = f'0{obj[1]}'
            
    except Exception as e:
        print(date)
    
    return int(f'{obj[2]}{obj[0]}{obj[1]}')

In [65]:
df_rates.date = df_rates.date.apply(lambda x:x.replace('-',"/"))
df_rates.date = df_rates.date.apply(lambda x:date_formatting(x))

In [66]:
df_rates.tail()

,date,4 weeks coupon equivalent,8 weeks coupon equivalent,13 weeks coupon equivalent,17 weeks coupon equivalent,26 weeks coupon equivalent,52 weeks coupon equivalent
6018,20260108,3.62,3.62,3.60,3.60,3.59,3.48
6019,20260107,3.61,3.59,3.60,3.60,3.59,3.48
6020,20260106,3.61,3.60,3.61,3.59,3.59,3.48
6021,20260105,3.63,3.63,3.62,3.61,3.59,3.47
6022,20260102,3.64,3.64,3.62,3.63,3.61,3.47


In [67]:
df_rates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6023 entries, 0 to 6022
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   date                        6023 non-null   int64  
 1   4 weeks coupon equivalent   6021 non-null   float64
 2   8 weeks coupon equivalent   1820 non-null   float64
 3   13 weeks coupon equivalent  6022 non-null   float64
 4   17 weeks coupon equivalent  818 non-null    float64
 5   26 weeks coupon equivalent  6022 non-null   float64
 6   52 weeks coupon equivalent  4421 non-null   float64
dtypes: float64(6), int64(1)
memory usage: 329.5 KB


In [68]:
def compute_continous_rate(df):
    for col in df.columns:
        if col == 'date':
            continue
        period = int(col.split(' ')[0])
        df[f'risk_free_rate_{period}'] = 365*np.log(1+(7*period*df[col])/(100*365))/(7*period)
        
    

In [69]:
compute_continous_rate(df_rates)

In [70]:
df_rates = df_rates.sort_values("date", ascending=True)

In [71]:
df_rates[df_rates.date==20250729]

,date,4 weeks coupon equivalent,8 weeks coupon equivalent,13 weeks coupon equivalent,17 weeks coupon equivalent,26 weeks coupon equivalent,52 weeks coupon equivalent,risk_free_rate_4,risk_free_rate_8,risk_free_rate_13,risk_free_rate_17,risk_free_rate_26,risk_free_rate_52
5861,20250729,4.33,4.35,4.35,4.34,4.24,4.09,0.043228,0.043355,0.043266,0.043096,0.041958,0.040088


In [72]:
df_rates.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6023 entries, 0 to 6004
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   date                        6023 non-null   int64  
 1   4 weeks coupon equivalent   6021 non-null   float64
 2   8 weeks coupon equivalent   1820 non-null   float64
 3   13 weeks coupon equivalent  6022 non-null   float64
 4   17 weeks coupon equivalent  818 non-null    float64
 5   26 weeks coupon equivalent  6022 non-null   float64
 6   52 weeks coupon equivalent  4421 non-null   float64
 7   risk_free_rate_4            6021 non-null   float64
 8   risk_free_rate_8            1820 non-null   float64
 9   risk_free_rate_13           6022 non-null   float64
 10  risk_free_rate_17           818 non-null    float64
 11  risk_free_rate_26           6022 non-null   float64
 12  risk_free_rate_52           4421 non-null   float64
dtypes: float64(12), int64(1)
memory usage:

In [73]:
df.head()

,Ticker,TradeDate,Strike,ExpiryDate,CallPut,LastTradePrice,BidPrice,AskPrice,BidImpliedVolatility,AskImpliedVolatility,...,AdjStrike,AbsMoneyness,Moneyness,FmtTradeDate,FmtExpiryDate,days_to_expiry,AdjExpiry,ExpirySpot,Symbol,Month_To_Expiry
0,AAPL,20100121,290.0,20100717,p,3.0000,2.9446,3.0000,0.2646,0.3438,...,10.3571,139.375059,140.0,2010-01-21,2010-07-17,177,2010-07-16,8.9250,SP,6
1,AAPL,20100121,280.0,20100717,p,2.6736,2.6143,2.6839,0.2845,0.3562,...,10.0000,134.569579,130.0,2010-01-21,2010-07-17,177,2010-07-16,8.9250,SP,6
2,AAPL,20100121,270.0,20100717,p,2.3464,2.3304,2.3482,0.3371,0.3509,...,9.6429,129.764100,130.0,2010-01-21,2010-07-17,177,2010-07-16,8.9250,SP,6
3,AAPL,20100121,130.0,20110122,c,2.9986,2.9679,3.0268,0.4075,0.4553,...,4.6429,62.479310,60.0,2010-01-21,2011-01-22,366,2011-01-21,11.6686,SP,12
4,AAPL,20100121,250.0,20100717,p,1.7493,1.7357,1.7429,0.3460,0.3502,...,8.9286,120.151794,120.0,2010-01-21,2010-07-17,177,2010-07-16,8.9250,SP,6


In [74]:
df = df.merge(df_rates, how='left', left_on='TradeDate', right_on='date')
df = df.drop(columns='date')

In [75]:
df.columns

Index(['Ticker', 'TradeDate', 'Strike', 'ExpiryDate', 'CallPut',
       'LastTradePrice', 'BidPrice', 'AskPrice', 'BidImpliedVolatility',
       'AskImpliedVolatility', 'OpenInterest', 'Volume', 'Delta', 'Gamma',
       'Vega', 'Theta', 'Rho', 'Spot', 'AdjSpot', 'AdjStrike', 'AbsMoneyness',
       'Moneyness', 'FmtTradeDate', 'FmtExpiryDate', 'days_to_expiry',
       'AdjExpiry', 'ExpirySpot', 'Symbol', 'Month_To_Expiry',
       '4 weeks coupon equivalent', '8 weeks coupon equivalent',
       '13 weeks coupon equivalent', '17 weeks coupon equivalent',
       '26 weeks coupon equivalent', '52 weeks coupon equivalent',
       'risk_free_rate_4', 'risk_free_rate_8', 'risk_free_rate_13',
       'risk_free_rate_17', 'risk_free_rate_26', 'risk_free_rate_52'],
      dtype='object')

In [76]:
len(df.columns)

41

In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 683323 entries, 0 to 683322
Data columns (total 41 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   Ticker                      683323 non-null  object        
 1   TradeDate                   683323 non-null  int64         
 2   Strike                      683323 non-null  float64       
 3   ExpiryDate                  683323 non-null  int64         
 4   CallPut                     683323 non-null  object        
 5   LastTradePrice              683323 non-null  float64       
 6   BidPrice                    683323 non-null  float64       
 7   AskPrice                    683323 non-null  float64       
 8   BidImpliedVolatility        683323 non-null  float64       
 9   AskImpliedVolatility        683323 non-null  float64       
 10  OpenInterest                683323 non-null  float64       
 11  Volume                      683323 non-

In [78]:
df[df.Theta.isnull()].Month_To_Expiry.value_counts()

Month_To_Expiry
0    813
Name: count, dtype: int64

In [79]:
df.columns

Index(['Ticker', 'TradeDate', 'Strike', 'ExpiryDate', 'CallPut',
       'LastTradePrice', 'BidPrice', 'AskPrice', 'BidImpliedVolatility',
       'AskImpliedVolatility', 'OpenInterest', 'Volume', 'Delta', 'Gamma',
       'Vega', 'Theta', 'Rho', 'Spot', 'AdjSpot', 'AdjStrike', 'AbsMoneyness',
       'Moneyness', 'FmtTradeDate', 'FmtExpiryDate', 'days_to_expiry',
       'AdjExpiry', 'ExpirySpot', 'Symbol', 'Month_To_Expiry',
       '4 weeks coupon equivalent', '8 weeks coupon equivalent',
       '13 weeks coupon equivalent', '17 weeks coupon equivalent',
       '26 weeks coupon equivalent', '52 weeks coupon equivalent',
       'risk_free_rate_4', 'risk_free_rate_8', 'risk_free_rate_13',
       'risk_free_rate_17', 'risk_free_rate_26', 'risk_free_rate_52'],
      dtype='object')

In [80]:
! pip install xlsxwriter


In [81]:
df.TradeDate.unique()

array([20100121, 20100129, 20100218, 20100226, 20100318, 20100331,
       20100422, 20100430, 20100520, 20100528, 20100617, 20100630,
       20100715, 20100730, 20100819, 20100831, 20100916, 20100930,
       20101014, 20101029, 20101118, 20101130, 20101216, 20101231,
       20110120, 20110131, 20110217, 20110228, 20110317, 20110331,
       20110414, 20110429, 20110519, 20110531, 20110616, 20110630,
       20110714, 20110729, 20110818, 20110831, 20110915, 20110930,
       20111020, 20111031, 20111117, 20111130, 20111215, 20111230,
       20120119, 20120131, 20120216, 20120229, 20120315, 20120330,
       20120426, 20120430, 20120517, 20120531, 20120614, 20120629,
       20120719, 20120731, 20120816, 20120831, 20120920, 20120928,
       20121018, 20121031, 20121115, 20121130, 20121220, 20121231,
       20130117, 20130131, 20130214, 20130228, 20130314, 20130328,
       20130418, 20130430, 20130516, 20130531, 20130620, 20130628,
       20130718, 20130731, 20130815, 20130830, 20130919, 20130

In [82]:
df.head()

,Ticker,TradeDate,Strike,ExpiryDate,CallPut,LastTradePrice,BidPrice,AskPrice,BidImpliedVolatility,AskImpliedVolatility,...,13 weeks coupon equivalent,17 weeks coupon equivalent,26 weeks coupon equivalent,52 weeks coupon equivalent,risk_free_rate_4,risk_free_rate_8,risk_free_rate_13,risk_free_rate_17,risk_free_rate_26,risk_free_rate_52
0,AAPL,20100121,290.0,20100717,p,3.0000,2.9446,3.0000,0.2646,0.3438,...,0.06,NaN,0.14,0.3,0.0002,NaN,0.0006,NaN,0.0014,0.002996
1,AAPL,20100121,280.0,20100717,p,2.6736,2.6143,2.6839,0.2845,0.3562,...,0.06,NaN,0.14,0.3,0.0002,NaN,0.0006,NaN,0.0014,0.002996
2,AAPL,20100121,270.0,20100717,p,2.3464,2.3304,2.3482,0.3371,0.3509,...,0.06,NaN,0.14,0.3,0.0002,NaN,0.0006,NaN,0.0014,0.002996
3,AAPL,20100121,130.0,20110122,c,2.9986,2.9679,3.0268,0.4075,0.4553,...,0.06,NaN,0.14,0.3,0.0002,NaN,0.0006,NaN,0.0014,0.002996
4,AAPL,20100121,250.0,20100717,p,1.7493,1.7357,1.7429,0.3460,0.3502,...,0.06,NaN,0.14,0.3,0.0002,NaN,0.0006,NaN,0.0014,0.002996


In [83]:
df.columns

Index(['Ticker', 'TradeDate', 'Strike', 'ExpiryDate', 'CallPut',
       'LastTradePrice', 'BidPrice', 'AskPrice', 'BidImpliedVolatility',
       'AskImpliedVolatility', 'OpenInterest', 'Volume', 'Delta', 'Gamma',
       'Vega', 'Theta', 'Rho', 'Spot', 'AdjSpot', 'AdjStrike', 'AbsMoneyness',
       'Moneyness', 'FmtTradeDate', 'FmtExpiryDate', 'days_to_expiry',
       'AdjExpiry', 'ExpirySpot', 'Symbol', 'Month_To_Expiry',
       '4 weeks coupon equivalent', '8 weeks coupon equivalent',
       '13 weeks coupon equivalent', '17 weeks coupon equivalent',
       '26 weeks coupon equivalent', '52 weeks coupon equivalent',
       'risk_free_rate_4', 'risk_free_rate_8', 'risk_free_rate_13',
       'risk_free_rate_17', 'risk_free_rate_26', 'risk_free_rate_52'],
      dtype='object')

In [84]:
df =df[['Ticker', 'TradeDate', 'ExpiryDate','CallPut','AdjStrike','Strike','AdjSpot','Spot',
        'Month_To_Expiry', 'days_to_expiry','ExpirySpot','AdjExpiry', 
       'Moneyness','AbsMoneyness', 'Symbol', 'LastTradePrice', 'BidPrice', 'AskPrice',
       'BidImpliedVolatility', 'AskImpliedVolatility',
       'OpenInterest', 'Volume', 'Delta', 'Gamma', 'Vega', 'Theta', 'Rho',
       'FmtTradeDate', 'FmtExpiryDate', '4 weeks coupon equivalent','8 weeks coupon equivalent', '13 weeks coupon equivalent',
       '17 weeks coupon equivalent', '26 weeks coupon equivalent',
       '52 weeks coupon equivalent', 'risk_free_rate_4', 'risk_free_rate_8',
       'risk_free_rate_13', 'risk_free_rate_17', 'risk_free_rate_26',
       'risk_free_rate_52']]

In [85]:
len(df.columns)

41

In [86]:
os.getcwd()

'c:\\Users\\shyam\\Dp Lrn\\Options'

In [87]:
df.to_csv(os.path.join(os.getcwd(),'Data','master_data.csv'),index=False)